# Gravitational Wave Templates, Matched Filtering, and Parameter Estimation

## Overview

This notebook continues from `data_tutorial.ipynb`, where we whitened the strain data
and saw the GW signal by eye. Here we move from qualitative detection to **quantitative
analysis**:

1. **Waveform models** — generate theoretical templates in the time and frequency domains,
   and build intuition for how the chirp parameters are encoded in the waveform shape
2. **Matched filtering** — introduce the noise-weighted inner product and develop
   step-by-step intuition for why time alignment is critical for extracting the SNR
3. **Parameter estimation** — set up a short Bayesian PE run with bilby and examine
   the resulting posterior distribution

> **Prerequisites**: Run `data_tutorial.ipynb` first and save the data files, or
> download the data in the Setup cell below.

### 🧪 Interactive exercises

| Exercise | Topic | After section |
|---|---|---|
| **Exercise 1** | Effect of spin on waveform duration | §1 Waveform models |
| **Exercise 2.a** | Time-alignment intuition: whitened overlap at specific offsets | §2 Matched filtering |
| **Exercise 2.b** | SNR sensitivity to a template mass error | §2 Matched filtering |
| **🏠 Homework** | PE with free inclination — the $d_L$–$\iota$ degeneracy | §3 Parameter estimation |

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import scipy.signal
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

from gwosc.datasets import event_gps
from gwosc.api import fetch_event_json
from gwpy.timeseries import TimeSeries
from pycbc.waveform import get_td_waveform, get_fd_waveform

plt.rcParams.update({"figure.figsize": (12, 5), "font.size": 12})
plt.style.use("seaborn-v0_8-whitegrid")
print("Packages loaded.")


In [ ]:
# ── Load data (from local files saved in data_tutorial, or re-download) ───────
gps = event_gps('GW250114_082203')
segment = (int(gps) - 16, int(gps) + 16)
event_time_in_segment = 16.0 + (gps - int(gps))

import os
if os.path.exists("GW250114_082203_H1.txt") and os.path.exists("GW250114_082203_L1.txt"):
    print("Loading data from local files...")
    hdata = TimeSeries.read("GW250114_082203_H1.txt")
    ldata = TimeSeries.read("GW250114_082203_L1.txt")
else:
    print("Downloading data from GWOSC (this may take a minute)...")
    hdata = TimeSeries.fetch_open_data('H1', *segment)
    ldata = TimeSeries.fetch_open_data('L1', *segment)

print(f"GPS time: {gps}")
print(f"H1: {len(hdata)} samples at {1/hdata.dt.value:.0f} Hz")
print(f"Event position in segment: {event_time_in_segment:.3f} s")


In [ ]:
# ── Reproduce ASD estimation and whitening from data_tutorial ─────────────────
def bandpass_window(freqs, f_low, f_high, taper_low=10., taper_high=100.):
    w = np.ones_like(freqs)
    ml = (freqs > f_low) & (freqs < f_low + taper_low)
    w[ml] = 0.5 * (1 - np.cos(np.pi * (freqs[ml] - f_low) / taper_low))
    w[freqs <= f_low] = 0.
    mh = (freqs > f_high - taper_high) & (freqs < f_high)
    w[mh] = 0.5 * (1 + np.cos(np.pi * (freqs[mh] - (f_high - taper_high)) / taper_high))
    w[freqs >= f_high] = 0.
    return w

def whiten_bandpass(ts, asd_func, f_low=20, f_high=500, taper_low=10., taper_high=100.):
    N = len(ts); dt = ts.dt.value
    tukey = scipy.signal.windows.tukey(N, alpha=0.1)
    fd = np.fft.rfft(tukey * ts.value)
    freqs = np.fft.rfftfreq(N, d=dt)
    norm = asd_func(freqs) / np.sqrt(2 * dt)
    bp   = bandpass_window(freqs, f_low, f_high, taper_low, taper_high)
    td   = np.fft.irfft(bp * fd / norm, n=N)
    return dt * np.arange(len(td)), td, bp, freqs

asd_h = hdata.asd(fftlength=2., method="median")
asd_l = ldata.asd(fftlength=2., method="median")
asd_h_interp = interp1d(asd_h.frequencies.value, asd_h.value, bounds_error=False, fill_value=np.inf)
asd_l_interp = interp1d(asd_l.frequencies.value, asd_l.value, bounds_error=False, fill_value=np.inf)

times_h, h_wbp, _, _ = whiten_bandpass(hdata, asd_h_interp)
times_l, l_wbp, _, _ = whiten_bandpass(ldata, asd_l_interp)
print("Whitening complete. Ready to proceed.")


---
## 1. Waveform Models

### The IMR (Inspiral–Merger–Ringdown) framework

A compact binary coalescence produces three distinct phases in its gravitational-wave emission:

| Phase | Description | Timescale |
|-------|-------------|-----------|
| **Inspiral** | The two bodies spiral in, losing energy to GW emission. Frequency and amplitude grow slowly (chirp). | seconds to minutes |
| **Merger** | The two objects plunge together. The waveform reaches peak amplitude. | milliseconds |
| **Ringdown** | The merged remnant oscillates as a perturbed Kerr black hole. The signal decays exponentially. | < 100 ms |

Several families of waveform **approximants** exist, differing in how they model these phases:

- **Phenomenological** (IMRPhenom*): closed-form fits to NR. Fast, flexible. Used in most GWTC analyses.
- **EOB** (SEOB*): based on the effective-one-body Hamiltonian. More physically motivated.
- **NR surrogates** (NRSur7dq4): interpolate directly over NR waveforms. Most accurate but restricted parameter range.

We use `pycbc`'s `get_td_waveform` (time-domain) and `get_fd_waveform` (frequency-domain) to generate templates.


In [ ]:
# ── Fetch catalog parameters for GW250114 ─────────────────────────────────────
event_data = fetch_event_json('GW250114_082203')
event_key  = list(event_data['events'].keys())[0]
params     = event_data['events'][event_key]

m1_source = params['mass_1_source']
m2_source = params['mass_2_source']
redshift  = params['redshift']
chi_eff   = params['chi_eff']
dist_mpc  = params['luminosity_distance']

# Convert to detector-frame masses (cosmological redshift stretches the waveform)
m1_det = m1_source * (1.0 + redshift)
m2_det = m2_source * (1.0 + redshift)

print(f"Source-frame:    m1 = {m1_source:.1f}, m2 = {m2_source:.1f} Msun,  z = {redshift:.3f}")
print(f"Detector-frame:  m1 = {m1_det:.1f}, m2 = {m2_det:.1f} Msun  (×(1+z) = {1+redshift:.3f})")
print(f"chi_eff = {chi_eff:.3f},  d_L = {dist_mpc:.0f} Mpc")


In [ ]:
# ── Generate the GW250114 template and show the three IMR phases ───────────────
hp, hc = get_td_waveform(
    approximant="IMRPhenomD",
    mass1=m1_det, mass2=m2_det,
    spin1z=chi_eff, spin2z=chi_eff,
    delta_t=hdata.dt.value, f_lower=20.0, distance=dist_mpc,
)

t = np.array(hp.sample_times)    # merger at t = 0

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# ── Top: full waveform ────────────────────────────────────────────────────────
axes[0].plot(t, hp, color='C0', lw=0.8, label=r'$h_+$')
axes[0].plot(t, hc, color='C1', lw=0.8, alpha=0.6, label=r'$h_\times$')
axes[0].axvline(0, color='red', ls='--', alpha=0.6, label='Merger (t=0)')
axes[0].set_xlim(-1, 0.1)
axes[0].set_xlabel('Time (s)');  axes[0].set_ylabel('Strain')
axes[0].set_title('IMRPhenomD template — full waveform')
axes[0].legend()
axes[0].annotate('Inspiral\n(chirp)', xy=(-0.5, 0), fontsize=11, ha='center',
                 arrowprops=dict(arrowstyle='->', color='gray'), xytext=(-0.5, max(hp)*0.5))

# ── Bottom: zoom on final 0.3 s ───────────────────────────────────────────────
axes[1].plot(t, hp, color='C0', lw=1.0, label=r'$h_+$')
axes[1].axvspan(-0.1, -0.0125, alpha=0.10, color='pink', label='Inspiral window')
axes[1].axvspan(-0.0125, 0, alpha=0.10, color='gold', label='Plunge-merger window')
axes[1].axvspan(0, 0.02, alpha=0.10, color='green', label='Ringdown')
axes[1].axvline(0, color='red', ls='--', alpha=0.6)
axes[1].set_xlim(-0.1, 0.025)
axes[1].set_xlabel('Time relative to merger (s)');  axes[1].set_ylabel('Strain')
axes[1].set_title('Zoom: final inspiral, merger, and ringdown')
axes[1].legend()

plt.suptitle(f'GW250114 template — IMRPhenomD  '
             f'(m1={m1_det:.0f}, m2={m2_det:.0f} Msun det.-frame, d={dist_mpc:.0f} Mpc)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ── How total mass sets the merger frequency ──────────────────────────────────
# More massive binaries merge at lower frequencies and for shorter durations.
# This is because the GW frequency at merger scales as f_merger ~ c^3 / (G M)

mass_configs = [
    (1.4, 1.4,  'M_tot = 2.8 Msun  (NS-like mass range)',  'C2'),
    (33, 32,  'M_tot = 65 Msun  (GW250114)',            'C0'),
    (70, 70,  'M_tot = 140 Msun (heavy BBH, e.g., GW231123)',           'C3'),
]

fig, axes = plt.subplots(len(mass_configs), 1, figsize=(14, 10), sharex=True)

for ax, (m1, m2, label, color) in zip(axes, mass_configs):
    hp_ex, _ = get_td_waveform(
        approximant="IMRPhenomD", mass1=m1, mass2=m2,
        spin1z=0., spin2z=0., delta_t=1/4096., f_lower=20., distance=1.
    )
    t_ex = np.array(hp_ex.sample_times)   # merger at t = 0
    ax.plot(t_ex, hp_ex / np.abs(hp_ex).max(), color=color, lw=0.8)
    ax.axvline(0, color='red', ls='--', alpha=0.6)
    ax.set_title(label)
    ax.set_ylabel('Norm. strain')
    ax.set_xlim(-3.5, 0.12)

axes[-1].set_xlabel('Time relative to merger (s)')
plt.suptitle('Effect of total mass on the waveform\n'
             'Heavier systems: shorter signal, lower merger frequency, fewer cycles in band',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ── Frequency-domain amplitude: the characteristic chirp mass signature ────────
# In the inspiral regime the FD amplitude follows: |h̃(f)| ∝ M_chirp^(5/6) f^(-7/6)
# This is the post-Newtonian (leading-order) prediction.

chirp_configs = [
    (1.4,  1.4,  'M_chirp ≈  1.2 Msun  (NS-like mass range)',  'C2'),
    (33,  32,  'M_chirp ≈ 28.7 Msun  (GW250114)', 'C0'),
    (70,  70,  'M_chirp ≈ 43.5 Msun  (70+70 Msun)',    'C3'),
]

fig, ax = plt.subplots(figsize=(11, 6))

f_ref = np.logspace(np.log10(20), np.log10(400), 200)

for m1, m2, label, color in chirp_configs:
    hp_f, _ = get_td_waveform(approximant="IMRPhenomD", mass1=m1, mass2=m2,
                               spin1z=0., spin2z=0., delta_t=1/4096.,
                               f_lower=10., distance=500.)
    hp_fd = hp_f.to_frequencyseries()
    f_arr = np.array(hp_fd.sample_frequencies)
    mask  = (f_arr > 15) & (f_arr < 2048)
    ax.loglog(f_arr[mask], np.abs(hp_fd[mask]), color=color, lw=1.5, label=label)

# Reference f^{-7/6} power law
ax.loglog(f_ref, 3e-23 * (f_ref/100)**(-7/6), 'k--', lw=1.2, alpha=0.6,
          label=r'$\propto f^{-7/6}$ (inspiral PN)')

ax.set_xlim(15, 1500);  ax.set_ylim(1e-27, 1e-20)
ax.set_xlabel('Frequency (Hz)');  ax.set_ylabel(r'$|\tilde{h}(f)|$ (strain/Hz)')
ax.set_title('FD amplitude: more massive → stronger signal, lower merger frequency')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()


---
## 🧪 Exercise 1: How spin changes the waveform

The spin-orbit coupling in GR causes aligned spins to stabilise the orbit and anti-aligned spins to destabilise it:

- **Aligned spin** (`chi_eff > 0`): orbit is more stable → slower inspiral, **more cycles** in the 20–500 Hz band
- **Anti-aligned spin** (`chi_eff < 0`): orbit shrinks faster → **fewer cycles**

The number of cycles in band matters enormously for the SNR: every extra inspiral cycle adds coherent signal power to the matched filter.

**Your task:** generate three `IMRPhenomT` (notice we are changing the approximant!) templates with GW250114's masses (`m1_det`, `m2_det`) but with `chi_eff` = −0.5, 0.0, +0.5. Normalise each by its peak amplitude and overlay them aligned at merger (t = 0). Use `xlim(-1.5, 0.05)`.

- How many extra cycles does `chi_eff = +0.5` add compared to non-spinning?
- Does spin change the **peak amplitude** or only the duration?

> **Hint:** Call `get_td_waveform(approximant='IMRPhenomT', mass1=m1_det, mass2=m2_det, spin1z=chi, spin2z=chi, delta_t=hdata.dt.value, f_lower=20., distance=1.)` for each `chi`. The time axis is `np.array(hp_s.sample_times)` — merger is at t = 0.

<details>
<summary><b>💡 Solution — click to expand</b></summary>

```python
spin_configs = [
    (-0.5, 'C1', r'$\chi_\mathrm{eff} = -0.5$  (anti-aligned)'),
    ( 0.0, 'C0', r'$\chi_\mathrm{eff} = 0$     (non-spinning)'),
    (+0.5, 'C2', r'$\chi_\mathrm{eff} = +0.5$  (aligned)'),
]

fig, ax = plt.subplots(figsize=(14, 5))
for chi, color, label in spin_configs:
    hp_s, _ = get_td_waveform(
        approximant='IMRPhenomT', mass1=m1_det, mass2=m2_det,
        spin1z=chi, spin2z=chi,
        delta_t=hdata.dt.value, f_lower=20., distance=1.
    )
    t_s = np.array(hp_s.sample_times)   # merger at t=0
    ax.plot(t_s, hp_s / np.abs(hp_s).max(), color=color, lw=0.9, label=label)

ax.axvline(0, color='red', ls='--', alpha=0.6, label='Merger (t=0)')
ax.set_xlim(-0.8, 0.05)
ax.set_xlabel('Time relative to merger (s)')
ax.set_ylabel('Normalised strain')
ax.set_title('Spin-orbit effect: same masses as GW250114, different chi_eff')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
# Takeaway: aligned spin gives a longer inspiral (more cycles in band).
# The merger amplitude is nearly independent of chi_eff.
```

</details>

In [ ]:
# ── Exercise 1 — your workspace ───────────────────────────────────────────────
# Generate three templates with chi_eff = -0.5, 0.0, +0.5 and overlay them.
# Use the masses of GW250114 (m1_det, m2_det already in memory).

# for chi in [-0.5, 0.0, +0.5]:
#     hp_s, _ = # Generate the time-domain waveform with get_td_waveform
#     
#     t_s = np.array(hp_s.sample_times)   # PyCBC convention: merger at t = 0
#     # plot hp_s / np.abs(hp_s).max()  ← normalise so shapes are comparable

# Suggested xlim: (-0.8, 0.05)  — focus on the final inspiral cycles


---
## 2. Matched Filtering

### The noise-weighted inner product

The **noise-weighted inner product** between two time series $a$ and $b$ is:

$$(a \mid b) \equiv 4\,\Re \int_0^{\infty} \frac{\tilde{a}^*(f)\,\tilde{b}(f)}{S_n(f)}\,df$$

where $S_n(f)$ is the one-sided PSD of the detector noise. This inner product **down-weights
frequency bands where the noise is loud** and up-weights bands where it is quiet.

For data $d = n + h$ (noise $n$ plus signal $h$) and a template $T$, the **signal-to-noise ratio** when the template merger is placed at time $t_0$ is:

$$\rho(t_0) = \frac{(d \mid T_{t_0})}{\sigma}, \qquad \sigma \equiv \sqrt{(T \mid T)}$$

where $T_{t_0}$ is the template time-shifted so its merger falls at $t_0$, and $\sigma$ is the **template norm** — the SNR one would expect if the template perfectly matched the signal at exactly the right time.

### Roadmap for this section

We build intuition in three steps before presenting the full efficient algorithm:

1. **Compute $\sigma$** — the template norm sets the SNR scale. We also prepare the Fourier quantities needed by Step 3.

2. **Exercise 2.a** — compute $(d \mid T_{t_0})$ *by hand* at 5 specific time offsets. This makes the time-alignment problem concrete: at the wrong $t_0$, the oscillating template and signal dephase and the inner product collapses toward zero.

3. **The FFT trick** — a time shift $\Delta t$ in the time domain is a phase factor $e^{-2\pi i f \Delta t}$ in the frequency domain. This turns the problem of evaluating $(d \mid T_{t_0})$ for *all* $t_0$ into a **single inverse FFT**, going from $\mathcal{O}(N^2)$ to $\mathcal{O}(N \log N)$.

In [ ]:
# ── Step 1: Template setup and norm ──────────────────────────────────────────
# Compute σ = sqrt((T|T)) and prepare the FFT quantities needed later.
N      = len(hdata)
dt_val = hdata.dt.value
df     = 1.0 / (N * dt_val)
tukey  = scipy.signal.windows.tukey(N, alpha=0.1)

# Pad template to data length (PyCBC places the merger at t=0 of the hp array)
n_hp = len(hp)
template_padded = np.zeros(N)
template_padded[:n_hp] = hp.numpy()

# Template FFT with physical units (× dt_val for correct normalisation)
tmpl_fd_rfft = np.fft.rfft(tukey * template_padded) * dt_val
freqs_rfft   = np.fft.rfftfreq(N, d=dt_val)
asd_rfft     = asd_h_interp(freqs_rfft);   asd_rfft[0] = asd_rfft[1]

# Template norm: σ ≡ sqrt((T|T)) = sqrt( 4 ∫ |T̃(f)|² / S_n(f) df )
sigma = np.sqrt(4.0 * np.sum(np.abs(tmpl_fd_rfft)**2 / asd_rfft**2) * df)
print(f"Template norm  σ = {sigma:.2f}")
print(f"  (σ is the expected optimal SNR if the template perfectly matches the signal)")

# Pre-compute data FFT quantities — reused by the full SNR cell below
data_fd_full = np.fft.fft(tukey * hdata.value) * dt_val
freqs_full   = np.fft.fftfreq(N, d=dt_val)
asd_full     = asd_h_interp(np.abs(freqs_full));   asd_full[0] = asd_full[1]
print("\nAll template and data FFTs ready. Proceed to the whitened template and Exercise 2.a.")


In [ ]:
# ── Step 2: Estimate the merger time from the data, then align the template ────
# The catalog GPS time from GWOSC is rounded to 0.1 s, so `event_time_in_segment`
# sets t=0 at ~.2 s while the true merger is nearer ~.22 s. For this clean, loud
# event we can refine the merger time directly from the whitened strain: its peak
# amplitude is an excellent proxy for the coalescence time. (Step 3 will confirm
# this with the rigorous matched-filter estimate.)
search = (times_h >= event_time_in_segment - 0.1) & (times_h <= event_time_in_segment + 0.1)
merger_time_est = float(times_h[search][np.argmax(np.abs(h_wbp[search]))])
print(f"Catalog event time (rounded to 0.1 s): {event_time_in_segment:.4f} s")
print(f"Estimated merger from whitened data:   {merger_time_est:.4f} s")
print(f"Refinement:                            {(merger_time_est - event_time_in_segment)*1000:+.1f} ms\n")

# Place the template merger at the data-estimated merger time.
merger_sample  = int(-float(hp.start_time) / dt_val)   # merger index inside hp array
desired_sample = int(merger_time_est / dt_val)          # estimated merger time in samples
shift = desired_sample - merger_sample

tmpl_aligned = np.zeros(N)
if shift >= 0:
    tmpl_aligned[shift:shift + n_hp] = hp.numpy()[:N - shift]
else:
    tmpl_aligned[:n_hp + shift] = hp.numpy()[-shift:]

tmpl_ts = TimeSeries(tmpl_aligned, dt=dt_val, t0=0)
times_tmpl, tmpl_wbp, _, _ = whiten_bandpass(tmpl_ts, asd_h_interp)

# Phase-align the template to the data for the overlay. The template's coalescence
# phase is arbitrary, so a plain amplitude fit can nearly vanish when the template and
# data are in quadrature. Project the data onto the template AND its 90°-quadrature
# (Hilbert) partner — the matched-filter reconstruction — so it overlays the data
# regardless of the phase offset.
win    = (times_h >= merger_time_est - 0.15) & (times_h <= merger_time_est + 0.05)
tmpl_q = np.imag(scipy.signal.hilbert(tmpl_wbp))        # 90° phase-shifted template
a = np.dot(h_wbp[win], tmpl_wbp[win]) / np.dot(tmpl_wbp[win], tmpl_wbp[win])
b = np.dot(h_wbp[win], tmpl_q[win])   / np.dot(tmpl_q[win],   tmpl_q[win])
tmpl_wbp_scaled = a * tmpl_wbp + b * tmpl_q             # amplitude- and phase-matched

print(f"Whitened template computed. Best-fit phase: {np.degrees(np.arctan2(b, a)):+.0f}°")
print(f"Merger placed at estimated time: {merger_time_est:.4f} s")
print("Ready for Exercise 2.a — the exact SNR peak time will be found in Step 3.")


---
## 🧪 Exercise 2.a: Computing the SNR at specific time offsets

You now have all the ingredients to evaluate the **actual matched-filter SNR** at a specific merger time $t_0$. A time shift $\delta t = t_0 - t_\text{nat}$ in the time domain is a phase factor $e^{-2\pi i f\,\delta t}$ in the frequency domain. The template's coalescence phase is unknown, so we take the **magnitude** of the complex inner product (phase-maximised SNR) — exactly what Step 3 computes with a single iFFT:

$$\rho(t_0) = \frac{4\,\left|\displaystyle\sum_{k>0} \frac{\tilde{d}^*(f_k)\;\tilde{T}(f_k)\;e^{-2\pi i f_k\,\delta t}}{S_n(f_k)}\right|\,\Delta f}{\sigma}$$

where $t_\text{nat}$ is the merger time of `template_padded` in the segment, $\tilde{T}$ is `tmpl_fd_rfft`, and $\tilde{d}$ is the rfft of the (windowed) data.

> **Note on the reference time:** the GWOSC catalog GPS time is rounded to 0.1 s, so `event_time_in_segment` is ~20 ms early. In Step 2 we refined it to `merger_time_est` using the peak of the whitened strain, so the offsets below are measured from that data-driven estimate.

**Your task:** evaluate $\rho(t_0)$ at 5 offsets from `merger_time_est`: −100 ms, −50 ms, 0 ms, +20 ms, +50 ms. Then make two plots:

1. **Alignment overlay** (3 sub-panels for −100 ms, 0 ms, +50 ms): plot the whitened data (grey, faded) and the whitened template (coloured) on the same axes. Put the computed SNR in the panel title — you will *see* the overlap collapsing as the offset grows.

2. **SNR vs offset scatter**: 5 diamond markers showing $\rho(t_0)$ vs offset. This is a preview of the full SNR time series that Step 3 will compute for *all* offsets at once.

> **Hint — key quantities already in memory:**
> - `data_rfft = data_fd_full[:len(freqs_rfft)]` (rfft = first N//2+1 elements of the full FFT)
> - `t_natural = -float(hp.start_time)` — merger time of `template_padded` in segment coords [s]
> - `delta_t = t0 - t_natural` where `t0 = merger_time_est + dt_ms/1000.`
> - `phase = np.exp(-2j * np.pi * freqs_rfft * delta_t)` — the time-shift phase factor
> - inner product (take the magnitude): `4.0 * np.sum(np.conj(data_rfft[1:]) * tmpl_fd_rfft[1:] * phase[1:] / asd_rfft[1:]**2) * df`, then `np.abs(...) / sigma`
> - For the overlay, shift the whitened template via `np.interp(times_h[mask], times_tmpl + dt_ms/1000., tmpl_wbp_scaled, left=0., right=0.)`

<details>
<summary><b>💡 Solution — click to expand</b></summary>

```python
data_rfft = data_fd_full[:len(freqs_rfft)]  # rfft(data) = fft(data)[:N//2+1]
t_natural = -float(hp.start_time)          # merger time of template_padded in segment [s]

offsets_ms = [-100, -50, 0, +20, +50]
snr_pts    = []

for dt_ms in offsets_ms:
    t0      = merger_time_est + dt_ms / 1000.   # offsets measured from the estimated merger
    delta_t = t0 - t_natural               # time shift from natural position [s]
    phase   = np.exp(-2j * np.pi * freqs_rfft * delta_t)
    # Complex matched-filter inner product; take |.| to phase-maximise (matches Step 3)
    inner   = 4.0 * np.sum(
        np.conj(data_rfft[1:]) * tmpl_fd_rfft[1:] * phase[1:] / asd_rfft[1:]**2
    ) * df
    snr_pts.append(np.abs(inner) / sigma)

print(f"{'Offset (ms)':>12}  {'SNR ρ(t0)':>12}")
print("─" * 28)
for dt_ms, rho in zip(offsets_ms, snr_pts):
    print(f"{dt_ms:>+12}  {rho:>12.1f}")

# ── Plot 1: alignment overlay at 3 representative offsets ─────────────────────
show_offsets = [-100, 0, +50]
show_colors  = ['C1', 'C0', 'C3']

trange = 0.15
mask   = (times_h >= merger_time_est - trange) & (times_h <= merger_time_est + trange)
t_plot = times_h[mask] - merger_time_est

fig, axes = plt.subplots(len(show_offsets), 1, figsize=(13, 8), sharex=True)
for ax, dt_ms, color in zip(axes, show_offsets, show_colors):
    ax.plot(t_plot, h_wbp[mask], color='gray', lw=0.6, alpha=0.4, label='H1 data (whitened)')
    tmpl_sh = np.interp(times_h[mask], times_tmpl + dt_ms / 1000.,
                        tmpl_wbp_scaled, left=0., right=0.)
    rho_here = snr_pts[offsets_ms.index(dt_ms)]
    ax.plot(t_plot, tmpl_sh, color=color, lw=1.4,
            label=f'Template  offset = {dt_ms:+d} ms   →   ρ = {rho_here:.1f}')
    ax.axvline(0, color='red', ls='--', alpha=0.5, lw=0.9)
    ax.set_ylabel('Whitened strain')
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(True, alpha=0.25)
axes[-1].set_xlabel('Time from estimated merger time (s)')
plt.suptitle('Template–data alignment at three time offsets', fontweight='bold')
plt.tight_layout()

# ── Plot 2: SNR vs time offset ─────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(9, 4))
ax2.scatter(offsets_ms, snr_pts, color='C0', s=100, zorder=5, marker='D',
            label='Computed offsets')
for dt_ms, rho in zip(offsets_ms, snr_pts):
    ax2.annotate(f'ρ = {rho:.1f}', xy=(dt_ms, rho),
                 xytext=(dt_ms + 3, rho + 1.5), fontsize=9, color='C0')
ax2.axvline(0, color='red', ls='--', alpha=0.5, label='Estimated merger time')
ax2.set_xlabel('Template time offset (ms)')
ax2.set_ylabel(r'SNR $\rho(t_0)$')
ax2.set_title('SNR at 5 specific offsets — Step 3 evaluates ALL offsets via one iFFT')
ax2.set_ylim(bottom=0)
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.tight_layout()
```

</details>


In [ ]:
data_rfft = data_fd_full[:len(freqs_rfft)]  # rfft(data) = fft(data)[:N//2+1]
t_natural = -float(hp.start_time)          # merger time of template_padded in segment [s]

offsets_ms = [-100, -50, 0, +20, +50]
snr_pts    = []

for dt_ms in offsets_ms:
    t0      = merger_time_est + dt_ms / 1000.   # offsets measured from the estimated merger
    delta_t = t0 - t_natural               # time shift from natural position [s]
    phase   = np.exp(-2j * np.pi * freqs_rfft * delta_t)
    # Complex matched-filter inner product; take |.| to phase-maximise (matches Step 3)
    inner   = 4.0 * np.sum(
        np.conj(data_rfft[1:]) * tmpl_fd_rfft[1:] * phase[1:] / asd_rfft[1:]**2
    ) * df
    snr_pts.append(np.abs(inner) / sigma)

print(f"{'Offset (ms)':>12}  {'SNR ρ(t0)':>12}")
print("─" * 28)
for dt_ms, rho in zip(offsets_ms, snr_pts):
    print(f"{dt_ms:>+12}  {rho:>12.1f}")

# ── Plot 1: alignment overlay at 3 representative offsets ─────────────────────
show_offsets = [-100, 0, +50]
show_colors  = ['C1', 'C0', 'C3']

trange = 0.15
mask   = (times_h >= merger_time_est - trange) & (times_h <= merger_time_est + trange)
t_plot = times_h[mask] - merger_time_est

fig, axes = plt.subplots(len(show_offsets), 1, figsize=(13, 8), sharex=True)
for ax, dt_ms, color in zip(axes, show_offsets, show_colors):
    ax.plot(t_plot, h_wbp[mask], color='gray', lw=0.6, alpha=0.4, label='H1 data (whitened)')
    tmpl_sh = np.interp(times_h[mask], times_tmpl + dt_ms / 1000.,
                        tmpl_wbp_scaled, left=0., right=0.)
    rho_here = snr_pts[offsets_ms.index(dt_ms)]
    ax.plot(t_plot, tmpl_sh, color=color, lw=1.4,
            label=f'Template  offset = {dt_ms:+d} ms   →   ρ = {rho_here:.1f}')
    ax.axvline(0, color='red', ls='--', alpha=0.5, lw=0.9)
    ax.set_ylabel('Whitened strain')
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(True, alpha=0.25)
axes[-1].set_xlabel('Time from estimated merger time (s)')
plt.suptitle('Template–data alignment at three time offsets', fontweight='bold')
plt.tight_layout()

# ── Plot 2: SNR vs time offset ─────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(9, 4))
ax2.scatter(offsets_ms, snr_pts, color='C0', s=100, zorder=5, marker='D',
            label='Computed offsets')
for dt_ms, rho in zip(offsets_ms, snr_pts):
    ax2.annotate(f'ρ = {rho:.1f}', xy=(dt_ms, rho),
                 xytext=(dt_ms + 3, rho + 1.5), fontsize=9, color='C0')
ax2.axvline(0, color='red', ls='--', alpha=0.5, label='Estimated merger time')
ax2.set_xlabel('Template time offset (ms)')
ax2.set_ylabel(r'SNR $\rho(t_0)$')
ax2.set_title('SNR at 5 specific offsets — Step 3 evaluates ALL offsets via one iFFT')
ax2.set_ylim(bottom=0)
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.tight_layout()

In [ ]:
# ── Exercise 2.a — your workspace ────────────────────────────────────────────
# Key arrays already in memory from Steps 1–2:
#   tmpl_fd_rfft, freqs_rfft, asd_rfft, sigma, df
#   data_fd_full, hp.start_time, merger_time_est
#   times_tmpl, tmpl_wbp_scaled, times_h, h_wbp

# ── A: compute SNR at 5 time offsets ─────────────────────────────────────────
data_rfft = data_fd_full[:len(freqs_rfft)]  # rfft = first N//2+1 elements of full FFT
t_natural = -float(hp.start_time)          # natural merger time in segment coords [s]

offsets_ms = [-100, -50, 0, +20, +50]
snr_pts    = []

for dt_ms in offsets_ms:
    t0      = merger_time_est + dt_ms / 1000.   # offsets from the data-estimated merger
    delta_t = t0 - t_natural
    # phase = np.exp(-2j * np.pi * freqs_rfft * delta_t)   # time shift → phase factor
    # inner = 4.0 * np.sum(
    #     np.conj(data_rfft[1:]) * tmpl_fd_rfft[1:] * phase[1:] / asd_rfft[1:]**2
    # ) * df
    # snr_pts.append(np.abs(inner) / sigma)   # magnitude → phase-maximised SNR

# ── B: alignment overlay (3 panels: -100ms, 0ms, +50ms) ─────────────────────
# mask = (times_h >= merger_time_est - 0.15) & (times_h <= merger_time_est + 0.15)
# For each panel: plot h_wbp[mask] (grey) and the shifted tmpl_wbp_scaled (coloured)
# Shift template: np.interp(times_h[mask], times_tmpl + dt_ms/1000., tmpl_wbp_scaled, ...)
# Put the computed SNR in each panel title

# ── C: SNR vs offset scatter ─────────────────────────────────────────────────
# plt.scatter(offsets_ms, snr_pts, marker='D', s=100)
# Annotate each point with its ρ value
# Add a vertical red dashed line at offset = 0

---
### Zooming in: how sharp is the SNR peak?

The coarse 5-point grid already hinted that the SNR falls off quickly away from the merger. Let's make that precise by scanning a **fine grid** of time offsets around `merger_time_est`.

Two things will stand out:

- The peak is only a **few milliseconds wide** — so the exact merger time matters a lot. A timing error of just 10–20 ms throws away most of the SNR.
- Our data-driven estimate `merger_time_est` (the whitened-amplitude peak) is close, but not *exactly* the matched-filter maximum — the fine scan reveals the true peak sits a couple of ms away and is somewhat higher than $\rho(0)$.

This is precisely the motivation for the next section: instead of guessing $t_0$ and scanning by hand, the **matched-filter SNR time series** evaluates $\rho(t)$ at *every* sample simultaneously with a single iFFT, and its maximum pins down both the merger time and the peak SNR.


In [ ]:
# ── Fine time scan of the SNR around the estimated merger ─────────────────────
# Same phase-shifted inner product as Exercise 2.a, but on a dense grid of offsets.
data_rfft = data_fd_full[:len(freqs_rfft)]
t_natural = -float(hp.start_time)

fine_offsets_ms = np.arange(-25., 25.001, 0.2)   # dense grid [ms] around merger_time_est
snr_fine = []
for dt_ms in fine_offsets_ms:
    t0      = merger_time_est + dt_ms / 1000.
    delta_t = t0 - t_natural
    phase   = np.exp(-2j * np.pi * freqs_rfft * delta_t)
    inner   = 4.0 * np.sum(
        np.conj(data_rfft[1:]) * tmpl_fd_rfft[1:] * phase[1:] / asd_rfft[1:]**2
    ) * df
    snr_fine.append(np.abs(inner) / sigma)
snr_fine = np.array(snr_fine)

# Key points on the curve
i_best   = np.argmax(snr_fine)
dt_best  = fine_offsets_ms[i_best]
rho_best = snr_fine[i_best]
rho_at0  = snr_fine[np.argmin(np.abs(fine_offsets_ms))]   # ρ at the estimate (0 ms)

# Approximate FWHM: width where ρ exceeds half the fine-grid maximum
half   = rho_best / 2.0
in_fwhm = fine_offsets_ms[snr_fine >= half]
fwhm    = in_fwhm.max() - in_fwhm.min() if in_fwhm.size else np.nan

print(f"ρ at the estimate (0 ms):        {rho_at0:.1f}")
print(f"Fine-grid peak:                  ρ = {rho_best:.1f}  at {dt_best:+.2f} ms")
print(f"SNR lost by being at 0 ms:       {(1 - rho_at0/rho_best)*100:.0f}%")
print(f"Approx. FWHM of the SNR peak:    {fwhm:.1f} ms")

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(fine_offsets_ms, snr_fine, color='C0', lw=1.8, label='fine scan  ρ(t₀)')
# Overlay the coarse 5-point grid from the exercise (those within range)
in_range = [(o, s) for o, s in zip(offsets_ms, snr_pts) if -25 <= o <= 25]
if in_range:
    ax.scatter([o for o, _ in in_range], [s for _, s in in_range],
               color='C3', s=90, marker='D', zorder=5,
               label='coarse 5-point grid (too sparse)')
ax.scatter([dt_best], [rho_best], color='green', s=110, zorder=6,
           label=f'true peak: ρ = {rho_best:.1f} at {dt_best:+.2f} ms')
ax.axvline(0, color='red', ls='--', alpha=0.6, label='estimated merger (0 ms)')
ax.axhline(half, color='gray', ls=':', alpha=0.7, label='half-maximum')
ax.set_xlabel('Template time offset from estimated merger (ms)')
ax.set_ylabel(r'SNR $\rho(t_0)$')
ax.set_title('Fine time scan: the SNR peak is only a few ms wide\n'
             'Even a small timing error discards a large fraction of the SNR')
ax.set_ylim(bottom=0)
ax.legend(fontsize=9, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
### Step 3: From manual time-shifting to all possible time shifts — the FFT trick

In Exercise 2.a you evaluated $(d \mid T_{t_0})$ at 5 specific offsets. Each evaluation looped over the full data array, so doing this for **every** possible $t_0$ (all $N \approx 131\,000$ samples) naively costs $\mathcal{O}(N^2)$ — far too slow for a real search.

**The key identity:** a time shift $\Delta t$ in the time domain is a pure phase factor in the frequency domain:

$$T(t - \Delta t) \;\xrightarrow{\;\mathcal{F}\;}\; \tilde{T}(f)\,e^{-2\pi i f \Delta t}$$

Substituting $t_0 \equiv \Delta t$ into the one-sided inner product:

$$(d \mid T_{t_0}) = 4\,\Re \int_0^\infty \frac{\tilde{d}^*(f)\,\tilde{T}(f)}{S_n(f)}\,e^{-2\pi i f t_0}\,df$$

The integral of $K(f) \equiv 2\,\tilde{d}^*(f)\tilde{T}(f)/S_n(f)$ swept over all $t_0$ is the **inverse Fourier transform** of $K$. Therefore:

$$\rho(t_0) = \frac{|\mathcal{F}^{-1}[K](t_0)|}{\sigma}$$

One iFFT — $\mathcal{O}(N \log N)$ instead of $\mathcal{O}(N^2)$ — delivers the full **SNR time series** for all merger times simultaneously. Its peak gives the best-estimate merger time and the maximum SNR.

In [ ]:
# ── Step 3: Full SNR time series via a single inverse FFT ─────────────────────
# Inner-product kernel on the full two-sided frequency grid.
# The factor 2 (instead of 4) accounts for both positive and negative frequencies.
tmpl_fd_full = np.fft.fft(tukey * template_padded) * dt_val
kernel   = 2.0 * np.conj(data_fd_full) * tmpl_fd_full / asd_full**2 * df

# One iFFT gives ρ(t0) = |(d|T_{t0})| / σ for every possible t0 simultaneously
snr_ts   = np.abs(np.fft.fft(kernel)) / sigma
times_mf = dt_val * np.arange(N) - float(hp.start_time)

peak_idx  = np.argmax(snr_ts)
peak_snr  = float(snr_ts[peak_idx])
peak_time = float(times_mf[peak_idx])

print(f"Peak SNR               = {peak_snr:.1f}")
print(f"Peak time              = {peak_time:.4f} s")
print(f"Published event time   = {event_time_in_segment:.4f} s")
print(f"Timing offset          = {(peak_time - event_time_in_segment)*1000:.1f} ms")

# ── Refine the whitened template to the actual SNR peak (for the plots below) ─
_ms  = int(-float(hp.start_time) / dt_val)
_ds  = int(peak_time / dt_val)
_sh  = _ds - _ms

_tmpl = np.zeros(N)
if _sh >= 0:
    _tmpl[_sh:_sh + n_hp] = hp.numpy()[:N - _sh]
else:
    _tmpl[:n_hp + _sh] = hp.numpy()[-_sh:]

_ts_ref = TimeSeries(_tmpl, dt=dt_val, t0=0)
times_tmpl, tmpl_wbp, _, _ = whiten_bandpass(_ts_ref, asd_h_interp)
# Phase-align the template to the data (its coalescence phase is arbitrary): project the
# data onto the template and its 90°-quadrature (Hilbert) partner — the matched-filter
# reconstruction — so the overlay is amplitude- AND phase-matched.
_win  = (times_h >= peak_time - 0.15) & (times_h <= peak_time + 0.05)
_tq   = np.imag(scipy.signal.hilbert(tmpl_wbp))
_a    = np.dot(h_wbp[_win], tmpl_wbp[_win]) / np.dot(tmpl_wbp[_win], tmpl_wbp[_win])
_b    = np.dot(h_wbp[_win], _tq[_win])      / np.dot(_tq[_win],      _tq[_win])
tmpl_wbp_scaled = _a * tmpl_wbp + _b * _tq
print(f"\ntimes_tmpl and tmpl_wbp_scaled refined to SNR peak. Best-fit phase: {np.degrees(np.arctan2(_b, _a)):+.0f}°")


In [ ]:
# ── Time-alignment intuition: SNR vs. template placement ─────────────────────
# The SNR time series rho(t) = SNR you would get if the merger occurred at time t.
# Here we visualise what data–template overlap looks like at three alignments.
# All offsets are measured relative to peak_time (the true SNR peak).

t_center = event_time_in_segment   # reference for the x-axis (red dashed line)
zoom_half = 0.13

offsets = {
    r'$-$100 ms  (too early)':  -0.100,
    r'$-$50 ms   (still off)':  -0.050,
    r'Correct alignment (peak)':  0.000,
}
colors = ['C1', 'C3', 'C0']

fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)
fig.subplots_adjust(hspace=0.45)

# Panel 1: whitened data only
ax0 = axes[0]
ax0.plot(times_h - t_center, h_wbp, color='gwpy:ligo-hanford', lw=0.7)
ax0.axvline(0, color='red', ls='--', alpha=0.8, label='True merger time (catalog)')
ax0.set_title('H1 whitened data only', fontsize=11)
ax0.set_xlim(-zoom_half, zoom_half)
ax0.legend(fontsize=9)
ax0.set_ylabel('Whitened strain')

# The whitened template's merger peak is at 'peak_time' in absolute segment coordinates.
# Offset_s shifts the template relative to that optimal placement.
# For the SNR: rho(peak_time + offset_s) = SNR if merger occurred at that displaced time.
for ax, (label, offset_s), color in zip(axes[1:], offsets.items(), colors):
    # Data (faded background)
    ax.plot(times_h - t_center, h_wbp,
            color='gwpy:ligo-hanford', lw=0.6, alpha=0.45, label='H1 data')
    # Template shifted by offset_s (shift the time axis only — valid approximation)
    ax.plot(times_tmpl + offset_s - t_center, tmpl_wbp_scaled,
            color=color, lw=1.5, alpha=0.9, label=f'Template: {label}')
    ax.axvline(0,              color='red',   ls='--', alpha=0.45)
    ax.axvline(peak_time - t_center + offset_s, color=color, ls=':', alpha=0.7,
               label='Template merger time')

    # Read SNR at the corresponding alignment from the precomputed time series
    t_query = peak_time + offset_s          # absolute time of template merger
    snr_here = float(np.interp(t_query, times_mf, snr_ts))
    frac = snr_here / peak_snr * 100
    ax.set_title(f'{label}  →  SNR = {snr_here:.1f}  ({frac:.0f}% of peak)',
                 fontsize=11)
    ax.legend(fontsize=9, loc='upper left')
    ax.set_xlim(-zoom_half, zoom_half)
    ax.set_ylabel('Whitened strain')

axes[-1].set_xlabel('Time from published merger time (s)')
plt.suptitle(
    'Matched filtering: SNR depends on template–signal alignment\n'
    'The SNR time series simultaneously evaluates all possible merger times',
    fontsize=12, fontweight='bold', y=1.01,
)
plt.show()


In [ ]:
# ── Plot the full SNR time series ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)

axes[0].plot(times_mf, snr_ts, color='C0', lw=0.7)
axes[0].axvline(event_time_in_segment, color='red', ls='--', alpha=0.7, label='Published event time')
axes[0].set_ylabel(r'SNR $\rho(t)$')
axes[0].set_title('Full SNR time series — H1  (each point = SNR if merger occurred at that time)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(times_mf, snr_ts, color='C0', lw=0.9)
axes[1].axvline(event_time_in_segment, color='red', ls='--', alpha=0.7, label='Published event time')
axes[1].axhline(peak_snr, color='green', ls=':', alpha=0.7, label=f'Peak SNR = {peak_snr:.1f}')
axes[1].scatter([peak_time], [peak_snr], color='green', zorder=5, s=80,
                label=f't_peak = {peak_time:.4f} s')
axes[1].set_xlim(event_time_in_segment - 0.20, event_time_in_segment + 0.14)
axes[1].set_ylim(0, peak_snr * 1.25)
axes[1].set_xlabel('Time (s from start of segment)')
axes[1].set_ylabel(r'SNR $\rho(t)$')
axes[1].set_title('Zoom: the peak marks the best-estimate merger time\n'
                  '(annotated points correspond to the three alignment panels above)')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

# Annotate the three demo offsets — offsets relative to peak_time
demo_labels = ['−100 ms', '−50 ms', 'peak (0 ms)']
demo_offsets = [-0.100, -0.050, 0.000]
demo_colors  = ['C1', 'C3', 'C0']
for offset_s, lbl, color in zip(demo_offsets, demo_labels, demo_colors):
    t_pt  = peak_time + offset_s
    snr_pt = float(np.interp(t_pt, times_mf, snr_ts))
    axes[1].scatter([t_pt], [snr_pt], color=color, s=70, zorder=6, marker='D')
    axes[1].annotate(f'{lbl}\nSNR={snr_pt:.1f}',
                     xy=(t_pt, snr_pt),
                     xytext=(t_pt + 0.01, snr_pt + 4.5 - demo_offsets.index(offset_s)*2),
                     fontsize=8, color=color,
                     arrowprops=dict(arrowstyle='->', color=color, lw=1.0))

plt.suptitle('GW250114_082203 — Matched-filter SNR time series (H1)', fontweight='bold')
plt.tight_layout()
plt.show()


---
## 🧪 Exercise 2.b: SNR sensitivity to a template mass error

The SNR time series tells us the best possible SNR **at every time lag** — but only if the template **shape** is correct. A wrong-mass template accumulates a phase error over the many inspiral cycles and the inner product partially cancels at every lag. No time-shifting can recover it: this is exactly why we need parameter estimation.

**Your task:** compute the SNR time series for a template with `mass1 = m1_det * 1.1` (10% heavier primary, everything else the same) and compare the peak SNR with `peak_snr`.

Steps:
1. Generate the wrong-mass template with `get_td_waveform`
2. Pad it to length `N` (same pattern as `template_padded`)
3. Compute its template norm `sigma_w` and cross-correlation kernel using `asd_rfft`, `data_fd_full`, `asd_full` — the pattern is identical to the SNR cell above; just use different variable names
4. Print the fractional SNR loss `(peak_snr - peak_snr_w) / peak_snr` and overlay both time series

> **Hint:** the only lines that differ from the Step 3 SNR cell are the `get_td_waveform` call and the variable names. The normalisations (`sigma_w = sqrt(4 * sum(|T_fd|² / ASD²) * df)`) and the cross-correlation kernel are identical.

<details>
<summary><b>💡 Solution — click to expand</b></summary>

```python
# Wrong-mass template: primary 10% heavier
hp_w, _ = get_td_waveform(
    approximant='IMRPhenomD',
    mass1=m1_det * 1.1, mass2=m2_det,   # ← only this changes
    spin1z=chi_eff, spin2z=chi_eff,
    delta_t=dt_val, f_lower=20., distance=dist_mpc,
)
n_w = len(hp_w)
tmpl_w = np.zeros(N)
tmpl_w[:min(n_w, N)] = hp_w.numpy()[:min(n_w, N)]

# Template FD (with physical dt factor) and norm
tmpl_w_fd_r = np.fft.rfft(tukey * tmpl_w) * dt_val
sigma_w     = np.sqrt(4. * np.sum(np.abs(tmpl_w_fd_r)**2 / asd_rfft**2) * df)

# SNR time series via FFT-based cross-correlation
tmpl_w_fd_f = np.fft.fft(tukey * tmpl_w) * dt_val
kernel_w    = 2. * np.conj(data_fd_full) * tmpl_w_fd_f / asd_full**2 * df
snr_ts_w    = np.abs(np.fft.fft(kernel_w)) / sigma_w

peak_snr_w = float(snr_ts_w.max())
snr_loss   = (peak_snr - peak_snr_w) / peak_snr * 100

print(f"Correct template peak SNR :  {peak_snr:.1f}")
print(f"Wrong mass (×1.1) peak SNR:  {peak_snr_w:.1f}")
print(f"Fractional SNR loss:         {snr_loss:.1f}%")
print()
print("This is why PE is needed: it finds the masses that maximise the SNR.")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(times_mf, snr_ts,   color='C0', lw=0.8,
        label=f'Correct template  (peak ρ = {peak_snr:.1f})')
ax.plot(times_mf, snr_ts_w, color='C1', lw=0.8, ls='--',
        label=f'm₁ × 1.1  (peak ρ = {peak_snr_w:.1f},  loss = {snr_loss:.0f}%)')
ax.axvline(event_time_in_segment, color='red', ls='--', alpha=0.6,
           label='Published event time')
ax.set_xlim(event_time_in_segment - 0.3, event_time_in_segment + 0.2)
ax.set_ylim(0, peak_snr * 1.2)
ax.set_xlabel('Time (s from start of segment)')
ax.set_ylabel(r'SNR $\rho(t)$')
ax.set_title('SNR loss from wrong template: even a 10% mass error matters')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
```

</details>

In [ ]:
# ── Exercise 2.b — your workspace ─────────────────────────────────────────────
# Generate a wrong-mass template (m1_det * 1.1) and compute its SNR time series.
# All needed arrays are already in memory:
#   N, dt_val, df, tukey, data_fd_full, asd_full, asd_rfft, peak_snr, times_mf

# Step 1: generate wrong-mass template
# hp_w, _ = get_td_waveform(
#     approximant='IMRPhenomD',
#     mass1=m1_det * 1.1, mass2=m2_det,   # ← only this changes
#     spin1z=chi_eff, spin2z=chi_eff,
#     delta_t=dt_val, f_lower=20., distance=dist_mpc,
# )

# Step 2: pad to length N (same pattern as template_padded)
# n_w = len(hp_w)
# tmpl_w = np.zeros(N)
# tmpl_w[:min(n_w, N)] = hp_w.numpy()[:min(n_w, N)]

# Step 3: compute sigma_w and snr_ts_w (same FFT pattern as Step 3 above)

# Step 4: compare peak SNRs and overlay the two time series


In [ ]:
# ── PyCBC cross-check: phase-maximised SNR ────────────────────────────────────
from pycbc.filter import matched_filter
from pycbc.psd import interpolate, inverse_spectrum_truncation
from pycbc.types import TimeSeries as PyCBCTimeSeries

strain_pycbc = PyCBCTimeSeries(hdata.value, delta_t=hdata.dt.value, epoch=hdata.t0.value)

# Resize template to data length and cyclic-shift merger to t=0 in the array
template_pycbc = PyCBCTimeSeries(np.zeros(N), delta_t=dt_val, epoch=strain_pycbc.start_time)
template_pycbc.data[:n_hp] = hp.numpy()
template_pycbc = template_pycbc.cyclic_time_shift(hp.start_time)

psd_pycbc = strain_pycbc.psd(4)
psd_pycbc = interpolate(psd_pycbc, strain_pycbc.delta_f)
psd_pycbc = inverse_spectrum_truncation(psd_pycbc, int(4 * strain_pycbc.sample_rate), low_frequency_cutoff=15.)

snr_pycbc = matched_filter(template_pycbc, strain_pycbc, psd=psd_pycbc, low_frequency_cutoff=20.)
snr_pycbc = snr_pycbc.crop(8, 4)

times_pycbc  = np.array([float(t - strain_pycbc.start_time) for t in snr_pycbc.sample_times])
peak_pycbc   = abs(snr_pycbc).data.argmax()
snr_pk_pycbc = float(abs(snr_pycbc[peak_pycbc]))
t_pk_pycbc   = float(times_pycbc[peak_pycbc])

print(f"Manual SNR peak:  {peak_snr:.1f}  at {peak_time:.4f} s")
print(f"PyCBC  SNR peak:  {snr_pk_pycbc:.1f}  at {t_pk_pycbc:.4f} s")
print(f"Difference:       Δρ = {abs(snr_pk_pycbc - peak_snr):.2f},  Δt = {(t_pk_pycbc - peak_time)*1000:.1f} ms")


In [ ]:
# Compare custom SNR with PyCBC's matched_filter output
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(times_mf, snr_ts, color='C0', lw=0.7, label='Custom SNR')
ax.plot(times_pycbc, abs(snr_pycbc), color='C1', lw=0.7, label='PyCBC SNR')
ax.axvline(event_time_in_segment, color='red', ls='--', alpha=0.7 , label='Published event time')
ax.set_xlim(event_time_in_segment - 0.20, event_time_in_segment + 0.14)
ax.set_ylim(0, peak_snr * 1.25)
ax.set_xlabel('Time (s from start of segment)')
ax.set_ylabel(r'SNR $\rho(t)$')
ax.set_title('Matched-filter SNR time series comparison: custom vs. PyCBC', fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── Template overlay on whitened data (best-fit alignment) ────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
t_start = event_time_in_segment - 0.15
t_end   = event_time_in_segment + 0.06

ax.plot(times_h, h_wbp, color='gwpy:ligo-hanford', lw=0.8, alpha=0.7,
        label='H1 whitened data')
ax.plot(times_tmpl, tmpl_wbp_scaled, color='black', lw=1.4, alpha=0.9,
        label=f'Best-fit template (peak SNR = {peak_snr:.1f})')
ax.axvline(event_time_in_segment, color='red', ls='--', alpha=0.5, label='Published event time')
ax.set_xlim(t_start, t_end)
ax.set_xlabel('Time (s from start of segment)')
ax.set_ylabel('Whitened strain')
ax.set_title('GW250114_082203 — best-fit template overlaid on whitened H1 data', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## 3. Parameter Estimation with bilby

Matched filtering tells us **that** a signal is present and approximately **when** it arrived. To measure the **source properties** (masses, spins, distance, sky location), we use **Bayesian parameter estimation**:

$$p(\theta \mid d) \propto \mathcal{L}(d \mid \theta)\,\pi(\theta)$$

where $\mathcal{L}(d \mid \theta)$ is the likelihood of the data given parameters $\theta$ and $\pi(\theta)$ is the prior.

For GW data, the log-likelihood under stationary Gaussian noise is:

$$\ln \mathcal{L}(d \mid \theta) = -\frac{1}{2}(d - h(\theta) \mid d - h(\theta))$$

i.e., the noise-weighted squared residual between data and template.

We use **[bilby](https://lscsoft.docs.ligo.org/bilby/)** with the **dynesty** nested sampler.
For this pedagogical run we fix the sky position, orientation, and spins at catalog
values and vary only $m_1$, $m_2$, and $d_L$ — keeping the run fast (~5–10 min).

> A pre-computed result is already available in `pe_short/GW250114_short_pe_result.json`
> from a previous run. Load it below or rerun the sampler if you want to reproduce it.


In [ ]:
import bilby
from bilby.core.prior import Uniform
from bilby.gw.conversion import convert_to_lal_binary_black_hole_parameters, generate_all_bbh_parameters
from pesummary.io import read as pe_read
import numpy as np

# ── Try to load the pre-computed result first ─────────────────────────────────
pe_result_path = "./pe_short/GW250114_short_pe_result.json"
if os.path.exists(pe_result_path):
    print(f"Loading existing PE result from {pe_result_path}")
    result_pe = bilby.result.read_in_result(outdir='pe_short', label='GW250114_short_pe')
    print("Loaded successfully.")
else:
    print("No pre-computed result found. Running the sampler (this takes ~5–10 min)...")
    print("Set up the likelihood first — see the cell below.")
    result_pe = None


In [ ]:
# ── bilby likelihood setup (run only if no pre-computed result) ───────────────
# Load published PE samples to fix sky position etc.
import h5py, pandas as pd

posterior_file = 'IGWN-GWTC5p0-29ebe06b7_25-GW250114_082203-combined_PEDataRelease.hdf5'
if not os.path.exists(posterior_file):
    print(f"Downloading PE data release to {posterior_file}...")
    import urllib.request
    url = ('https://zenodo.org/api/records/20348006/files/'
           'IGWN-GWTC5p0-29ebe06b7_25-GW250114_082203-combined_PEDataRelease.hdf5/content')
    urllib.request.urlretrieve(url, posterior_file)
    print("Done.")

with h5py.File(posterior_file, 'r') as f:
    samples_pe = pd.DataFrame.from_records(
        np.array(f['C00:IMRPhenomXPHM-SpinTaylor']['posterior_samples'][:])
    )

m1_det_pe = samples_pe['mass_1'].median()
m2_det_pe = samples_pe['mass_2'].median()
print(f"Catalog median detector-frame masses: m1 = {m1_det_pe:.2f}, m2 = {m2_det_pe:.2f} Msun")

# Now we fix the sky position and orientation parameters to the catalog median values.
ra_pe = samples_pe['ra'].median()
dec_pe = samples_pe['dec'].median()
theta_jn_pe = samples_pe['theta_jn'].median()
psi_pe = samples_pe['psi'].median()
print(f"Catalog median sky position: RA = {ra_pe:.3f} rad, Dec = {dec_pe:.3f} rad")

# We employ a short segment of data around the event for the PE run (4 s total, ±2 s around the merger).
pe_start = gps - 2;  pe_end = gps + 2
H1_pe = bilby.gw.detector.get_empty_interferometer('H1')
L1_pe = bilby.gw.detector.get_empty_interferometer('L1')
H1_pe.set_strain_data_from_gwpy_timeseries(hdata.crop(pe_start, pe_end))
L1_pe.set_strain_data_from_gwpy_timeseries(ldata.crop(pe_start, pe_end))

# We recompute the PSDs for this short segment, using a Tukey window with alpha = 2 * roll_off / 4.
psd_alpha = 2 * H1_pe.strain_data.roll_off / 4
H1_psd_ts = hdata.crop(gps - 16, gps + 16)
L1_psd_ts = ldata.crop(gps - 16, gps + 16)
H1_psd = H1_psd_ts.psd(fftlength=4, overlap=0, window=('tukey', psd_alpha), method='median')
L1_psd = L1_psd_ts.psd(fftlength=4, overlap=0, window=('tukey', psd_alpha), method='median')

H1_pe.power_spectral_density = bilby.gw.detector.PowerSpectralDensity(
    frequency_array=H1_psd.frequencies.value, psd_array=H1_psd.value)
L1_pe.power_spectral_density = bilby.gw.detector.PowerSpectralDensity(
    frequency_array=L1_psd.frequencies.value, psd_array=L1_psd.value)
for ifo in [H1_pe, L1_pe]:
    ifo.minimum_frequency = 20;  ifo.maximum_frequency = 1024

# We setup the priors for our analysis. We fix the sky position and orientation parameters to the catalog median values, and we use uniform priors for the masses, distance, time, and phase.
prior_pe = bilby.core.prior.PriorDict()
prior_pe['mass_1']             = Uniform(name='mass_1', minimum=m1_det_pe - 5., maximum=m1_det_pe + 5.)
prior_pe['mass_2']             = Uniform(name='mass_2', minimum=m2_det_pe - 5., maximum=m2_det_pe + 5.)
prior_pe['luminosity_distance']= Uniform(name='luminosity_distance', minimum=100., maximum=2000.)
prior_pe['geocent_time']       = Uniform(name='geocent_time', minimum=gps - 0.1, maximum=gps + 0.1)
prior_pe['phase']              = Uniform(name='phase', minimum=0, maximum=2 * np.pi)
prior_pe['ra']      = samples_pe['ra'].median()
prior_pe['dec']     = samples_pe['dec'].median()
prior_pe['theta_jn']= samples_pe['theta_jn'].median()
prior_pe['psi']     = samples_pe['psi'].median()
for k in ['a_1', 'a_2', 'tilt_1', 'tilt_2', 'phi_12', 'phi_jl']:
    prior_pe[k] = 0.

# We set up the waveform generator and likelihood for the PE run. We use the IMRPhenomD waveform approximant, which is a fast frequency-domain model for binary black hole coalescences.
wf_gen = bilby.gw.WaveformGenerator(
    duration=4, sampling_frequency=4096,
    frequency_domain_source_model=bilby.gw.source.lal_binary_black_hole,
    waveform_arguments=dict(waveform_approximant='IMRPhenomD',
                            reference_frequency=20., catch_waveform_errors=True),
    parameter_conversion=convert_to_lal_binary_black_hole_parameters,
)
likelihood_pe = bilby.gw.likelihood.GravitationalWaveTransient(
    [H1_pe, L1_pe], wf_gen, priors=prior_pe,
    time_marginalization=True, phase_marginalization=True, distance_marginalization=True,
)
print("Likelihood set up. Run the next cell to start sampling.")


In [ ]:
# ── Run the sampler (skip if result already loaded above) ─────────────────────
if result_pe is None:
    result_pe = bilby.run_sampler(
        likelihood_pe, prior_pe,
        sampler='dynesty', outdir='pe_short', label='GW250114_short_pe',
        conversion_function=generate_all_bbh_parameters,
        nlive=128, dlogz=0.1, sample='rwalk', nact=1, walks=1, clean=True,
    )
    print("Sampling complete.")


In [ ]:
# ── PE results: credible intervals and corner plot ────────────────────────────
pe_params = ['mass_1_source', 'mass_2_source', 'chirp_mass_source', 'luminosity_distance']
pe_params_avail = [p for p in pe_params if p in result_pe.posterior.columns]

print("90% credible intervals from the short PE run:")
print(f"  {'Parameter':<25s}  {'5%':>10}  {'Median':>10}  {'95%':>10}")
print("  " + "─" * 55)
for p in pe_params_avail:
    lo, med, hi = np.quantile(result_pe.posterior[p].values, [0.05, 0.5, 0.95])
    print(f"  {p:<25s}  {lo:>10.3f}  {med:>10.3f}  {hi:>10.3f}")

print()
print(f"log Bayes factor = {result_pe.log_bayes_factor:.1f} ± {result_pe.log_evidence_err:.1f}")

# Corner plot
result_pe.plot_corner(parameters=pe_params_avail, prior=True)


---
## 🏠 Homework Exercise: the distance–inclination degeneracy

### Why does inclination matter?

GW detectors measure a combination of signal amplitude and **inclination angle** $\iota$ — the angle between the binary's orbital angular momentum and the line of sight. The dominant polarisation amplitudes scale as:

$$h_+ \propto \frac{1 + \cos^2 \iota}{d_L}, \qquad h_\times \propto \frac{2 \cos \iota}{d_L}$$

A **face-on** binary ($\iota \approx 0°$, maximum emission) at large distance produces nearly the same measured strain as an **edge-on** binary ($\iota \approx 90°$, minimum emission) at a smaller distance. With a limited number of detectors, this degeneracy is only partially broken — the $d_L$–$\iota$ posterior is typically **banana-shaped**.

In the PE run above we fixed `theta_jn` to the catalog median to save time. Let's now see what happens when we let it vary.

### Your task

1. Copy `prior_pe` and replace `prior_pe['theta_jn']` with `bilby.core.prior.Sine(name='theta_jn')` — this is the physically motivated prior (uniform in $\cos\iota$, i.e. random orientation on the sky)
2. Re-run the sampler (may take 15–20 min with `nlive=256`)
3. Make a corner plot with `['mass_1_source', 'mass_2_source', 'luminosity_distance', 'theta_jn']`

**What to expect:** a clear anti-correlation in the $d_L$–$\theta_{JN}$ panel — face-on configurations ($\theta_{JN} \approx 0$ or $\pi$) are allowed at large distances, while edge-on requires a closer source. The mass posteriors should be nearly unchanged.

> **Optional extension:** also free `psi` (polarisation angle). Does it tighten or broaden the $d_L$ posterior?

<details>
<summary><b>💡 Setup code — click to expand</b></summary>

```python
import copy

prior_hw = copy.deepcopy(prior_pe)
prior_hw['theta_jn'] = bilby.core.prior.Sine(name='theta_jn')
# prior_hw['psi'] = bilby.core.prior.Uniform(0, np.pi, name='psi')  # optional

print("Modified prior — theta_jn is now free:")
print(" ", prior_hw['theta_jn'])

result_hw = bilby.run_sampler(
    likelihood_pe, prior_hw,
    sampler='dynesty', outdir='pe_short', label='GW250114_hw_pe',
    conversion_function=generate_all_bbh_parameters,
    nlive=256, dlogz=0.1, sample='rwalk', nact=2, walks=1,
)

params_hw    = ['mass_1_source', 'mass_2_source', 'luminosity_distance', 'theta_jn']
params_avail = [p for p in params_hw if p in result_hw.posterior.columns]
result_hw.plot_corner(parameters=params_avail, prior=True)
```

</details>

In [ ]:
# ── Homework Exercise — your workspace ────────────────────────────────────────
import copy

# Step 1: copy prior_pe and replace theta_jn with a Sine prior
# (Sine = uniform in cos(theta_jn), physically motivated for random orientation)
# prior_hw = copy.deepcopy(prior_pe)
# prior_hw['theta_jn'] = bilby.core.prior.Sine(name='theta_jn')
# print("theta_jn prior:", prior_hw['theta_jn'])

# Optional: also free the polarisation angle
# prior_hw['psi'] = bilby.core.prior.Uniform(0, np.pi, name='psi')

# Step 2: re-run the sampler (~15–20 min with nlive=256)
# result_hw = bilby.run_sampler(
#     likelihood_pe, prior_hw,
#     sampler='dynesty', outdir='pe_short', label='GW250114_hw_pe',
#     conversion_function=generate_all_bbh_parameters,
#     nlive=256, dlogz=0.1, sample='rwalk', nact=2, walks=1,
# )

# Step 3: corner plot — look for the d_L vs theta_jn anti-correlation
# params_hw = ['mass_1_source', 'mass_2_source', 'luminosity_distance', 'theta_jn']
# params_avail = [p for p in params_hw if p in result_hw.posterior.columns]
# result_hw.plot_corner(parameters=params_avail, prior=True)


---
## Summary

| Section | What we covered | Key result |
|---------|-----------------|------------|
| **Setup** | Loaded strain data and reproduced whitening | Ready to build templates |
| **1. Waveform models** | TD/FD waveforms, IMR phases, mass dependence | Heavier systems: lower $f_\mathrm{merger}$, shorter duration; $|\tilde{h}| \propto f^{-7/6}$ in the inspiral |
| **🧪 Exercise 1** | Spin effect on waveform duration | Aligned spin → more inspiral cycles in band; peak amplitude nearly unchanged |
| **2. Matched filtering** | Noise-weighted inner product, $\sigma$, SNR time series | SNR collapses to zero when the template is misaligned; peak $\rho \approx 50$ in H1 alone |
| **🧪 Exercise 2.a** | Whitened overlap at 5 specific time offsets | Overlap peaks at correct alignment and collapses at ±50 ms — intuition before the full SNR plot |
| **2. Time alignment** | Side-by-side overlay at −100 ms, −50 ms, and 0 ms offsets | Misalignment by 100 ms already cuts the SNR by >60% — timing precision matters! |
| **🧪 Exercise 2.b** | SNR loss from a 10% mass error | Wrong template → SNR drop at every lag — no time-shifting can compensate; motivates PE |
| **2. PyCBC cross-check** | Phase-maximised complex SNR | Manual and PyCBC results agree to < 2 SNR units |
| **3. PE** | bilby + dynesty on masses and distance | Posterior clearly localised; $m_1$–$m_2$ correlation reflects the chirp-mass degeneracy |
| **🏠 Homework** | PE with free inclination angle | Expected: banana-shaped $d_L$–$\theta_{JN}$ posterior from the amplitude–inclination degeneracy |

### Discussion questions

1. Why does the SNR drop so steeply when the template is misaligned by only 50–100 ms?
2. In the FD amplitude plot, where does the $f^{-7/6}$ law break down and why?
3. The PE run fixes sky position and spins. What would change if we freed those parameters?
4. The bilby corner plot shows $m_1$–$m_2$ correlation. How does this relate to what you see in the `catalog_tutorial.ipynb` corner plot?